In [5]:
import pandas as pd
df = pd.read_csv("../data/LLM_labeled.csv")

cols_to_drop = ['subreddit', 'type', 'score', 'created_utc']

df_2 = df.drop(columns=cols_to_drop)

In [6]:
global_df      = df_2.copy()
rioplatense_df = df_2[df_2["dialect"] == "rioplatense"].reset_index(drop=True)
cdmx_df        = df_2[df_2["dialect"] == "cdmx"].reset_index(drop=True)
madrid_df      = df_2[df_2["dialect"] == "madrid"].reset_index(drop=True)

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset
import numpy as np
import evaluate
import pandas as pd

# ── Labels 
label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label  = {0: 'negativo', 1: 'neutral', 2: 'positivo'}

# ── Model / tokenizer 
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)

# ── Metrics 
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


def prepare_dataset(df: pd.DataFrame, tokenizer, max_length: int = 128) -> Dataset:
    df = df[["text", "sentiment"]].copy().reset_index(drop=True)

    #  Normalise labels: strip whitespace + lowercase so mapping never KeyErrors
    df["sentiment"] = df["sentiment"].str.strip().str.lower()

    # Drop rows whose label isn't in the mapping (safety net)
    valid_mask = df["sentiment"].isin(label2id)
    dropped = (~valid_mask).sum()
    if dropped:
        print(f"Dropping {dropped} rows with unrecognised sentiment labels.")
    df = df[valid_mask].reset_index(drop=True)

    dataset = Dataset.from_pandas(df, preserve_index=False)

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_length)

    # Integer-encode labels in a separate pass so column types stay clean
    def encode_labels(examples):
        return {"labels": [label2id[l] for l in examples["sentiment"]]}

    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.map(encode_labels, batched=True)
    dataset = dataset.remove_columns(["text", "sentiment"])
    return dataset


def train_bert(df: pd.DataFrame, name: str, epochs: int = 3):
    print(f"\n{'='*55}")
    print(f"  Training: {name.upper()}  |  samples: {len(df)}")
    print(f"{'='*55}")

    dataset    = prepare_dataset(df, tokenizer)

    train_test = dataset.train_test_split(test_size=0.2, seed=42)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    args = TrainingArguments(
        output_dir=f"./{name}_bert",
        num_train_epochs=epochs,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        warmup_steps=100,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        report_to="none",
        save_total_limit=1,
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_test["train"],
        eval_dataset=train_test["test"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )

    train_output = trainer.train()
    trainer.save_model(f"{name}_bert_final")
    tokenizer.save_pretrained(f"{name}_bert_final")

    results = [x for x in trainer.state.log_history if "eval_accuracy" in x][-1]
    print(f"{name.upper()}: accuracy = {results['eval_accuracy']:.4f}")
    return results


# ── Train all 4 models ────────────────────────────────────────────────────────
global_results      = train_bert(global_df,      "global",      epochs=3)
rioplatense_results = train_bert(rioplatense_df, "rioplatense", epochs=4)
cdmx_results        = train_bert(cdmx_df,        "cdmx",        epochs=4)
madrid_results      = train_bert(madrid_df,      "madrid",      epochs=4)


  Training: GLOBAL  |  samples: 5293


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 30357.77it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECT

Epoch,Training Loss,Validation Loss,Accuracy
1,0.777449,0.698354,0.713881
2,0.473068,0.669223,0.726157
3,0.163607,0.892591,0.714825


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

✅ GLOBAL: accuracy = 0.7148

  Training: RIOPLATENSE  |  samples: 2455


...

✅ CDMX: accuracy = 0.6988

  Training: MADRID  |  samples: 398


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19524.06it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECT

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.085059,0.462500
2,No log,1.012329,0.500000
3,No log,0.969751,0.512500
4,No log,0.924715,0.550000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.23it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

✅ MADRID: accuracy = 0.5500


In [15]:
import transformers
print(transformers.__version__)

4.40.0


In [16]:
import sys
import torch
print(sys.executable)
print(torch.__file__)

c:\Users\Estanislao\Documents\VSCODE\Sentiment_analysis_tool\venv\Scripts\python.exe
c:\Users\Estanislao\Documents\VSCODE\Sentiment_analysis_tool\venv\Lib\site-packages\torch\__init__.py


In [2]:
import torch
import transformers

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

try:
    from transformers import AutoModel
    # This specifically triggers the PyTorch backend check
    model = AutoModel.from_pretrained("bert-base-uncased")
    print("✅ Success! Transformers 4.40 is now using PyTorch 2.3.0.")
except ImportError:
    print("❌ Still not seeing PyTorch. You may need to restart VS Code.")

c:\Users\Estanislao\Documents\VSCODE\Sentiment_analysis_tool\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.11.0+cu130
Transformers version: 5.3.0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6927.73it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Success! Transformers 4.40 is now using PyTorch 2.3.0.


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5060 Laptop GPU
